In [46]:
import pandas as pd
import re
import nltk

import warnings
warnings.filterwarnings("ignore")

nltk.download("stopwords")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Pandav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
data = pd.read_csv("spamham.csv")

data.head()

,Label,Message
0,ham,Subject: enron methanol ; meter # : 988291\nth...
1,ham,"Subject: hpl nom for january 9 , 2001\n( see a..."
2,ham,"Subject: neon retreat\nho ho ho , we ' re arou..."
3,spam,"Subject: photoshop , windows , office . cheap ..."
4,ham,Subject: re : indian springs\nthis deal is to ...


In [8]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

stopwords = stopwords.words("english")
ps = PorterStemmer()
corpus = []

for i in range(0,len(data)):
    review = re.sub("[^a-zA-Z]" , " " , data["Message"][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(i) for i in review if i not in stopwords]
    review = " ".join(review)
    corpus.append(review)

In [30]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=4000)

X = cv.fit_transform(corpus)

y = data["Label"].map({"ham" : 0 , "spam" : 1})

## Grid Search

- Why do we use Grid Search?

`GridSearchCV` is a technique to search through the best parameter values from the given set of the grid of parameters. It is basically a cross-validation method. the model and the parameters are required to be fed in. Best parameter values are extracted and then the predictions are made.

## Select the best model
- so here we have some list of the best text classification algorithms we imported. Now we will compare each model's score and see which model is performing better than rest of the others

# 1. Multinomial Naive Bayes Classifier

The multinomial NB classifier has a hyperparameter called **`alpha`**. It is the **smoothing parameter** to avoid **zero counts** when calculating the frequencies. 

For example, if we are now classifying a new SMS with a word "ryan" which never exist in the spam emails within our training dataset, the **likelihood** for this word will be zero. This will casue the **overall likelihood** to be zero (because we take the product of all **individual likelihoods**) for no matter what class of output variable we have.

Therefore, we need to add **additional counts** to each word when calculating the frequencies to avoid have a zero likelihood value. **Alpha** indicates how many **additional counts** we add.

# 2. Gaussian Naive Bayes Classifier

There is one hyperparameter we need to tune: **`var_smoothing`**. This is the **portion of the largest variance** of all features that is added to variances for **calculation stability**.

# 3. SVC
SVC, or Support Vector Classifier, is a supervised machine learning algorithm typically used for classification tasks. SVC works by mapping data points to a high-dimensional space and then finding the optimal hyperplane that divides the data into two classes.

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB , GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score , classification_report , confusion_matrix

models = {
    "MultinomialNB" : MultinomialNB(),
    "GaussianNB" : GaussianNB(),
    "SVC" : SVC()
}

In [35]:
#create a function to evalute models and return a report 

from sklearn.model_selection import train_test_split

def evaluate_models(X , y , models):

    X_train , X_test , y_train , y_test = train_test_split(X , y ,test_size=0.2 , random_state=42)

    model_list = []
    score_list = []

    for i in range(len(list(models))):
        
        model = list(models.values())[i]
        
        model.fit(X_train.toarray() , y_train)

        y_pred = model.predict(X_test.toarray())

        score = accuracy_score(y_test , y_pred)

        model_name = list(models.keys())[i]
        print(f"--------{model_name}---------\n")
        print(f"accuracy_score : {score}")

        model_list.append(model_name)
        score_list.append(score)
    print()

    report = pd.DataFrame()
    report["Model_name"] = model_list
    report["Score"] = score_list
    return report

In [36]:
evaluate_models(X , y , models)

--------MultinomialNB---------

accuracy_score : 0.8947368421052632
--------GaussianNB---------

accuracy_score : 0.9316281357599606
--------SVC---------

accuracy_score : 0.9286768322675848



,Model_name,Score
0,MultinomialNB,0.894737
1,GaussianNB,0.931628
2,SVC,0.928677


- ### From the report above we can see that the Gaussian Naive Bayes model performed the best, so we will continue training our model using Gaussian Naive Bayes algorithm.

In [37]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [39]:
import numpy as np

params = {
    "var_smoothing" : np.random.exponential(0.000000001 , 20)
}

gnb_model = GaussianNB()

gnb_cv = GridSearchCV(gnb_model , param_grid=params , cv = 10)

gnb_cv.fit(X_train.toarray() , y_train)

print(f"tuned hyperparameter : {gnb_cv.best_params_}")
print(f"accuracy_score : {gnb_cv.best_score_}")

tuned hyperparameter : {'var_smoothing': 3.811619443335121e-09}
accuracy_score : 0.9226229860820775


In [43]:
spam_detect_model = GaussianNB(**gnb_cv.best_params_)
spam_detect_model.fit(X_train.toarray() , y_train)

y_pred = spam_detect_model.predict(X_test.toarray())

print(f"accuracy_score : {accuracy_score(y_test , y_pred)}")
print(f"connfusion_matrix : \n {confusion_matrix(y_test , y_pred)}")

accuracy_score : 0.9326119035907526
connfusion_matrix : 
 [[1550   57]
 [  80  346]]


- So we can see that the model performed well and we have got an accuracy of 93% which is pretty insane. In our project we will be having all these models and we will be selecting the models based on the performence.